## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [14]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = sorted(set(sequence))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

#TODO: Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [15]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 100  # Increase length of each input sequence for better context
stride = 10             # Stride for creating sequences, can test lower or higher
embedding_dim = 30     # Increased dimension for character embeddings
hidden_size = 128      # Increase number of features in the hidden state of the RNN
learning_rate = 0.001  # Lower learning rate for stability
num_epochs = 10        # Increase number of epochs to allow for better learning
batch_size = 64        # Increase batch size for more efficient training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> Here

sequence_length: Determines the number of characters in each training input sequence. It sets the context size the RNN uses for making predictions.

stride: Controls how much the training window moves across the text data when creating sequences. Smaller strides increase the number of overlapping sequences, improving training data density.

embedding_dim: Specifies the size of the vector used to represent each character. Larger dimensions can capture more complex relationships but may increase computational cost.

hidden_size: Defines the number of hidden units in the RNN layer. Higher values allow the model to learn more complex patterns in the data.

num_layers: Indicates the number of RNN layers stacked together. Additional layers help the model capture hierarchical patterns in the data.

learning_rate: Determines the size of each update step during optimization. A smaller learning rate leads to slower but more stable convergence.

num_epochs: Specifies how many times the entire training dataset is passed through the model. More epochs allow the model to learn better but can risk overfitting.

batch_size: Sets the number of training samples processed together in one forward and backward pass. Larger batch sizes lead to faster computation but may require more memory.

vocab_size: Represents the total number of unique characters in the dataset, serving as the input and output dimensions of the model.

input_size: Matches the vocabulary size because each character is one-hot encoded for input to the RNN.

output_size: Matches the vocabulary size because the RNN outputs probabilities for each character in the vocabulary.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [16]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [17]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [10]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [18]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size, embedding_dim=embedding_dim).to(device)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [19]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/4379 [00:00<?, ?it/s]C:\Users\<redacted>\AppData\Local\Temp\ipykernel_26864\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\<redacted>\AppData\Local\Temp\ipykernel_26864\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 4379/4379 [12:04<00:00,  6.04it/s]


Epoch [1/10], Loss: 1.7569, Accuracy: 48.71%


Epoch 2/10: 100%|██████████| 4379/4379 [16:47<00:00,  4.34it/s]


Epoch [2/10], Loss: 1.5723, Accuracy: 53.90%


Epoch 3/10: 100%|██████████| 4379/4379 [11:08<00:00,  6.55it/s]


...

Epoch [8/10], Loss: 1.5179, Accuracy: 55.46%


Epoch 9/10: 100%|██████████| 4379/4379 [12:52<00:00,  5.67it/s]


Epoch [9/10], Loss: 1.5161, Accuracy: 55.51%


Epoch 10/10: 100%|██████████| 4379/4379 [09:56<00:00,  7.34it/s]

Epoch [10/10], Loss: 1.5144, Accuracy: 55.55%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [13]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [20]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    
    model.eval()  # Set the model to evaluation mode
    total_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    if len(test_loader) == 0:  # Check if the test_loader is empty
        print("Test loader is empty. Unable to calculate average loss and accuracy.")
    else:
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)  # Move data to device
            hidden = None  # Initialize hidden state for RNN

            # Forward pass
            logits, hidden = model(inputs, hidden)

            # Calculate loss
            loss = criterion(logits.view(-1, vocab_size), targets.view(-1))  # Reshape for compatibility
            total_loss += loss.item()

            # Get the predictions and compute accuracy
            _, predicted = torch.max(logits.view(-1, vocab_size), dim=1)  # Get index of the max logits
            correct_predictions += (predicted == targets.view(-1)).sum().item()  # Compare predictions to targets
            total_predictions += targets.numel()  # Total number of target elements

        # Calculate average loss and accuracy
        avg_loss = total_loss / len(test_loader) if len(test_loader) > 0 else 0  # Average loss over all batches
        accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0  # Calculate accuracy as a percentage

        print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

C:\Users\<redacted>\AppData\Local\Temp\ipykernel_26864\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\<redacted>\AppData\Local\Temp\ipykernel_26864\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.5721, Test Accuracy: 53.78%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [22]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits
    scaled_logits = logits / temperature  # Adjust logits based on temperature
    # Convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)  
    # Sample from probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0, device='cpu'):
    """
    Generate text using the trained RNN model.
    
    Args:
        model: The trained RNN model used for character prediction.
        start_text: The initial string provided by the user to start generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: A scaling factor for randomness in predictions. Default is 1.0.
        device: The device to run the model on ('cpu' or 'cuda').

    Returns:
        generated_text: The full generated sequence including original input and new characters.
    """
    model.eval()  # Set the model to evaluation mode
    start_text = start_text.lower()  # Normalize the input to lowercase
    
    # Convert the start text to indices
    try:
        input_indices = torch.tensor([char_to_idx[char] for char in start_text], dtype=torch.long).unsqueeze(0).to(device)  # Shape: [1, n]
    except KeyError as e:
        print(f"Error: Character '{e.args[0]}' not found in character mapping.")
        return ""  # Return empty string if there's an issue with the character

    generated_text = start_text  # Start with the given input text
    hidden = None  # Initialize hidden state (for the RNN)

    # Generate k additional characters
    for _ in range(k):
        # Forward pass through the model
        logits, hidden = model(input_indices, hidden)

        # Extract logits for the last character in the sequence
        last_logits = logits[:, -1, :]  # Shape: [1, vocab_size]
        
        # Sample the next character using the temperature scaling
        next_char_idx = sample_from_output(last_logits, temperature).item()  # Get the predicted index
        
        # Convert the index back to the character
        next_char = idx_to_char[next_char_idx]  # Map index to character

        # Append the predicted character to the generated text
        generated_text += next_char

        # Update input for the next prediction (include the newly predicted character)
        input_indices = torch.cat([input_indices, torch.tensor([[next_char_idx]], dtype=torch.long, device=device)], dim=1)  # Shape: [1, n+1]
        
        # Maintain the input size to the last 'n' characters as needed
        input_indices = input_indices[:, -n:]  # Keep only the last 'n' characters

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':  # Exit condition
        print("Exiting...")
        break
    
    n = len(start_text)  # Determine the length of the initial input
    k = int(input("Enter the number of characters to generate: "))  # Number of characters to generate
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0  # Default to 1.0 if none provided
    
    completed_text = generate_text(model, start_text, n, k, temperature, device)  # Generate the text
    print(f"Generated text: {completed_text}")  # Output the generated text
    

Training complete. Now you can generate text.
Error: Character '2' not found in character mapping.
Generated text: 
Generated text: and lif
Generated text: faith w
Generated text: faith peolly we


KeyboardInterrupt: Interrupted by user

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.

Answer:

Experiments & Observations:

Simple "abcdefghijklmnopqrstuvwxyz" * 100" Dataset:

1) Training: Achieved near-zero loss and almost 100% accuracy within a few epochs due to the dataset’s simplicity. The model easily memorized the exact sequence.
2) Generated Text: Perfect alphabetical sequences, no meaningful change with varying temperature since the pattern is trivial.


warandpeace.txt Dataset:

1) Training: Much larger and more complex text. After tuning parameters and reducing model size, final loss stabilized around 1.57 with ~53.8% accuracy—far from perfect due to the dataset’s complexity. Training took longer, and the model could not memorize the data like the simple dataset.
2) Generated Text: Character distributions resembled the original text, but outputs were mostly incoherent. Lower temperatures produced more predictable but repetitive sequences; higher temperatures introduced more variety but less coherence.


Reflections & Key Insights:

1) RNNs can easily memorize simple, repetitive data but struggle with complex, natural language.
2) Temperature strongly influences output randomness and diversity: low temperature yields more stable sequences, while high temperature increases variability and “creativity.”
